# Hybrid and Ensemble Models for Diabetes Readmission Prediction

## Objective

This notebook investigates two ensemble approaches:

1. Random Forest + XGBoost using soft voting.
2. XGBoost + DNN using stacking with Logistic Regression as the meta-learner.

The experiments reuse the preprocessed dataset and patient-grouped
data splits established in Notebook 01.

Validation data is used for model selection and threshold optimisation.
The test set is reserved for final evaluation.

## Research Objective

To investigate whether combining complementary machine-learning
and deep-learning models improves diabetes readmission prediction
compared with individual centralised models.

In [ ]:
import os
import json
import joblib
import random
import numpy as np
import pandas as pd

from pathlib import Path
from scipy import sparse

SEED = 42

np.random.seed(SEED)
random.seed(SEED)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Working directory:", Path.cwd())

In [ ]:
cwd = Path.cwd()

# Locate the previously exported DNN data directory
dnn_data_dir = None

for parent in [cwd, *cwd.parents]:
    candidate = parent / "models" / "artifacts" / "dnn_data"

    if candidate.exists():
        dnn_data_dir = candidate
        break

if dnn_data_dir is None:
    raise FileNotFoundError(
        "Could not locate models/artifacts/dnn_data. "
        "Please check the notebook location."
    )

artifacts_dir = dnn_data_dir.parent

print("DNN data directory:", dnn_data_dir)
print("Artifacts directory:", artifacts_dir)

print("\nFiles available in artifacts directory:")

for path in sorted(artifacts_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(artifacts_dir))

In [ ]:
required_files = [
    "X_train.npz",
    "X_validation.npz",
    "X_test.npz",
    "train_metadata.csv",
    "validation_metadata.csv",
    "test_metadata.csv",
    "preprocessor.joblib",
    "metadata.json"
]

for filename in required_files:
    file_path = dnn_data_dir / filename

    print(
        f"{filename}:",
        "Available" if file_path.exists() else "MISSING"
    )

In [ ]:
X_train = sparse.load_npz(dnn_data_dir / "X_train.npz").tocsr()
X_val = sparse.load_npz(dnn_data_dir / "X_validation.npz").tocsr()
X_test = sparse.load_npz(dnn_data_dir / "X_test.npz").tocsr()

train_metadata = pd.read_csv(dnn_data_dir / "train_metadata.csv")
val_metadata = pd.read_csv(dnn_data_dir / "validation_metadata.csv")
test_metadata = pd.read_csv(dnn_data_dir / "test_metadata.csv")

with open(dnn_data_dir / "metadata.json", "r") as f:
    metadata = json.load(f)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

print("\nMetadata columns:")
print(train_metadata.columns.tolist())

print("\nMetadata JSON:")
print(json.dumps(metadata, indent=2))

In [ ]:
# Inspect the metadata before selecting columns
display(train_metadata.head())

In [ ]:
y_train = train_metadata["target"].to_numpy(dtype=np.int64)
y_val = val_metadata["target"].to_numpy(dtype=np.int64)
y_test = test_metadata["target"].to_numpy(dtype=np.int64)

groups_train = train_metadata["patient_group"].to_numpy()
groups_val = val_metadata["patient_group"].to_numpy()
groups_test = test_metadata["patient_group"].to_numpy()

print("Training positive cases:", int(y_train.sum()))
print("Validation positive cases:", int(y_val.sum()))
print("Testing positive cases:", int(y_test.sum()))

print("\nTraining patients:", len(np.unique(groups_train)))
print("Validation patients:", len(np.unique(groups_val)))
print("Testing patients:", len(np.unique(groups_test)))

In [ ]:
assert X_train.shape[0] == len(y_train)
assert X_val.shape[0] == len(y_val)
assert X_test.shape[0] == len(y_test)

assert set(np.unique(y_train)).issubset({0, 1})
assert set(np.unique(y_val)).issubset({0, 1})
assert set(np.unique(y_test)).issubset({0, 1})

train_patients = set(groups_train)
val_patients = set(groups_val)
test_patients = set(groups_test)

assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)

print("All shape and target checks passed.")
print("No patient overlap between splits.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

import torch
import torch.nn as nn

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features=0.5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.7,
    colsample_bytree=0.9,
    min_child_weight=1,
    reg_lambda=10,
    scale_pos_weight=7.860991651909941,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

print("Training tuned Random Forest...")
rf_model.fit(X_train, y_train)

print("Training tuned XGBoost...")
xgb_model.fit(X_train, y_train)

print("Both tree models trained successfully.")

In [ ]:
rf_val_prob = rf_model.predict_proba(X_val)[:, 1]
xgb_val_prob = xgb_model.predict_proba(X_val)[:, 1]

print("Validation predictions generated.")

print("RF predictions:", rf_val_prob.shape)
print("XGBoost predictions:", xgb_val_prob.shape)

In [ ]:
reconstructed_results = pd.DataFrame([
    {
        "Model": "Random Forest Tuned",
        "ROC-AUC": roc_auc_score(y_val, rf_val_prob),
        "PR-AUC": average_precision_score(y_val, rf_val_prob)
    },
    {
        "Model": "XGBoost Tuned",
        "ROC-AUC": roc_auc_score(y_val, xgb_val_prob),
        "PR-AUC": average_precision_score(y_val, xgb_val_prob)
    }
])

display(reconstructed_results.round(4))

In [ ]:
previous_validation = {
    "Random Forest Tuned": 0.2417,
    "XGBoost Tuned": 0.2524
}

for _, row in reconstructed_results.iterrows():
    model_name = row["Model"]
    current_score = row["PR-AUC"]
    previous_score = previous_validation[model_name]

    print(f"\n{model_name}")
    print("Previous PR-AUC:", previous_score)
    print("Reconstructed PR-AUC:", round(current_score, 4))
    print("Absolute difference:", round(abs(current_score - previous_score), 6))

In [ ]:
model_dir = artifacts_dir / "hybrid_models"
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    rf_model,
    model_dir / "random_forest_tuned.joblib"
)

joblib.dump(
    xgb_model,
    model_dir / "xgboost_tuned.joblib"
)

print("Models saved successfully:")
print(model_dir / "random_forest_tuned.joblib")
print(model_dir / "xgboost_tuned.joblib")

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Selected device:", DEVICE)

In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class SparseIndexDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return idx


def make_sparse_collate(X, y):
    def collate_fn(indices):
        indices = np.asarray(indices, dtype=np.int64)

        # Convert only the current batch to dense float32
        X_batch = X[indices].toarray().astype(
            np.float32, copy=False
        )

        y_batch = y[indices].astype(
            np.float32, copy=False
        )

        return (
            torch.from_numpy(X_batch),
            torch.from_numpy(y_batch)
        )

    return collate_fn


print("Sparse batch dataset implementation ready.")

In [ ]:
BATCH_SIZE = 256

train_dataset = SparseIndexDataset(X_train, y_train)
val_dataset = SparseIndexDataset(X_val, y_val)
test_dataset = SparseIndexDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=make_sparse_collate(X_train, y_train)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=make_sparse_collate(X_val, y_val)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=make_sparse_collate(X_test, y_test)
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
import time

start = time.time()

batch_x, batch_y = next(iter(train_loader))

print("Batch shape:", batch_x.shape)
print("Label shape:", batch_y.shape)
print("Data type:", batch_x.dtype)
print("Batch loading time:", round(time.time() - start, 3), "seconds")

In [ ]:
import random
import copy
import time
import numpy as np
import torch
import torch.nn as nn

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Training device:", DEVICE)

negative_count = np.sum(y_train == 0)
positive_count = np.sum(y_train == 1)

POS_WEIGHT = negative_count / positive_count

print("Negative samples:", negative_count)
print("Positive samples:", positive_count)
print("Positive class weight:", POS_WEIGHT)

In [ ]:
class DiabetesReadmissionDNN(nn.Module):

    def __init__(self, input_dim, dropout_rate=0.30):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


INPUT_DIM = X_train.shape[1]

dnn_model = DiabetesReadmissionDNN(
    input_dim=INPUT_DIM,
    dropout_rate=0.30
).to(DEVICE)

print(dnn_model)
print("Input features:", INPUT_DIM)

In [ ]:
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [POS_WEIGHT],
        dtype=torch.float32,
        device=DEVICE
    )
)

optimizer = torch.optim.AdamW(
    dnn_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

MAX_EPOCHS = 50
PATIENCE = 7

print("Loss function:", criterion)
print("Optimizer:", optimizer)
print("Maximum epochs:", MAX_EPOCHS)
print("Early stopping patience:", PATIENCE)

In [ ]:
import time
import torch

features, labels = next(iter(train_loader))

start = time.time()

features = features.to(DEVICE)
labels = labels.to(DEVICE)

optimizer.zero_grad(set_to_none=True)

logits = dnn_model(features)
loss = criterion(logits, labels)

loss.backward()
optimizer.step()

if DEVICE.type == "mps":
    torch.mps.synchronize()

print("One training batch completed.")
print("Loss:", loss.item())
print("Batch training time:", round(time.time() - start, 3), "seconds")

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for features, labels in loader:

        features = features.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        logits = model(features)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


def evaluate_model(model, loader, criterion):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_probabilities = []
    all_labels = []

    with torch.no_grad():

        for features, labels in loader:

            features = features.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(features)

            loss = criterion(logits, labels)

            probabilities = torch.sigmoid(logits)

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            all_probabilities.extend(
                probabilities.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    all_probabilities = np.asarray(all_probabilities)
    all_labels = np.asarray(all_labels)

    avg_loss = total_loss / total_samples

    pr_auc = average_precision_score(
        all_labels,
        all_probabilities
    )

    roc_auc = roc_auc_score(
        all_labels,
        all_probabilities
    )

    return avg_loss, pr_auc, roc_auc, all_probabilities

In [ ]:
best_val_pr_auc = -np.inf
best_epoch = 0
epochs_without_improvement = 0

history = []

best_model_state = None

for epoch in range(1, MAX_EPOCHS + 1):

    epoch_start = time.time()

    train_loss = train_one_epoch(
        dnn_model,
        train_loader,
        optimizer,
        criterion
    )

    val_loss, val_pr_auc, val_roc_auc, val_probabilities = (
        evaluate_model(
            dnn_model,
            val_loader,
            criterion
        )
    )

    if DEVICE.type == "mps":
        torch.mps.synchronize()

    epoch_time = time.time() - epoch_start

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_pr_auc": val_pr_auc,
        "val_roc_auc": val_roc_auc,
        "epoch_time_seconds": epoch_time
    })

    print(
        f"Epoch {epoch}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val PR-AUC: {val_pr_auc:.4f} | "
        f"Val ROC-AUC: {val_roc_auc:.4f} | "
        f"Time: {epoch_time:.2f}s"
    )

    if val_pr_auc > best_val_pr_auc:

        best_val_pr_auc = val_pr_auc
        best_epoch = epoch

        best_model_state = copy.deepcopy(
            dnn_model.state_dict()
        )

        epochs_without_improvement = 0

    else:

        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}. "
            f"Best epoch: {best_epoch}"
        )

        break


dnn_model.load_state_dict(best_model_state)

print("\nDNN training completed.")
print("Best epoch:", best_epoch)
print("Best validation PR-AUC:", round(best_val_pr_auc, 4))

In [ ]:
import platform
import psutil
import torch

memory = psutil.virtual_memory()

print("Operating System:", platform.platform())
print("Processor:", platform.processor())
print("Architecture:", platform.machine())
print("Physical CPU cores:", psutil.cpu_count(logical=False))
print("Logical CPU cores:", psutil.cpu_count(logical=True))
print("Total RAM (GB):", round(memory.total / 1024**3, 2))
print("Available RAM (GB):", round(memory.available / 1024**3, 2))
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available())

if torch.cuda.is_available():
    print("NVIDIA GPU:", torch.cuda.get_device_name(0))

In [ ]:
import time
import torch
import numpy as np

# Use the existing training loader
features_cpu, labels_cpu = next(iter(train_loader))

# Start with a manageable batch size
BENCHMARK_BATCH_SIZE = 64

features_cpu = features_cpu[:BENCHMARK_BATCH_SIZE].contiguous()
labels_cpu = labels_cpu[:BENCHMARK_BATCH_SIZE].contiguous()


def synchronize_device(device):
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize(device)


def benchmark_device(device):

    torch.manual_seed(42)

    if device.type == "mps":
        torch.mps.manual_seed(42)

    model = DiabetesReadmissionDNN(
        input_dim=INPUT_DIM,
        dropout_rate=0.30
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.001,
        weight_decay=1e-4
    )

    criterion = torch.nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [POS_WEIGHT],
            dtype=torch.float32,
            device=device
        )
    )

    X = features_cpu.to(device)
    y = labels_cpu.to(device)

    def training_step():
        model.train()
        optimizer.zero_grad(set_to_none=True)

        logits = model(X)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        return loss

    # Warm-up
    training_step()
    synchronize_device(device)

    # Measure three training steps
    start = time.perf_counter()

    for _ in range(3):
        loss = training_step()

    synchronize_device(device)

    elapsed = time.perf_counter() - start

    result = {
        "Device": str(device),
        "Batch Size": BENCHMARK_BATCH_SIZE,
        "Average Step Time (seconds)": round(elapsed / 3, 4),
        "Final Loss": round(loss.item(), 5)
    }

    del model, optimizer, criterion, X, y

    if device.type == "mps":
        torch.mps.empty_cache()
    elif device.type == "cuda":
        torch.cuda.empty_cache()

    return result


devices = [torch.device("cpu")]

if torch.cuda.is_available():
    devices.append(torch.device("cuda"))

if torch.backends.mps.is_available():
    devices.append(torch.device("mps"))

benchmark_results = []

for device in devices:
    print(f"\nTesting {device}...")

    try:
        result = benchmark_device(device)
        benchmark_results.append(result)
        print(result)

    except Exception as error:
        print("Benchmark failed:", repr(error))

benchmark_df = pd.DataFrame(benchmark_results)
display(benchmark_df)